# `ptof_obs_liveness_detection`

## What this notebook does
Detects liveness and data-quality gaps across the pipeline:
1. **Capability silence** (WARN) — a registered capability hasn't produced output in >grace_hours
2. **Shift context missing** (WARN) — blank shift_date/shift_type/batch_nbr in output records
3. **ETL pipeline health** (CRITICAL, NEW) — an upstream ETL task failed, meaning the agent is
   running on stale source data even though it's still producing outputs

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `02_latency_detection` — runs after `01_bronze_projections`,
  before `06_alert`.
- **Upstream:** reads `v_llm_bronze`, `v_etl_bronze` (built by `ptof_obs_bronze_projection`),
  and `capability_registry` (human-curated by `ptof_obs_setup_seed`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `etl_pipeline_health` (CRITICAL) and checks
  `capability_silence` and `shift_context_missing` (WARN) via direct queries.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `v_etl_bronze`, `capability_registry`
- **Writes:** `capability_silence`, `shift_context_missing`, `etl_pipeline_health`

## Dropped detectors (prod migration 2026-09-10)
- `latency_anomalies` / `latency_anomaly_findings` — no `latency_ms` in prod
- `credential_fastfail_daily` — dev-specific model doesn't exist in prod
- `write_lag_daily` — depends on `latency_ms` for ingest-only computation
- `latency_failures` — depends on `success`, `error_class` (not in prod)
- `capability_health` / `capability_error_rate_alert` / `capability_error_rate_findings` — depends on `success`
- `prompt_size_drift` — no `user_prompt_chars` or `capability_latency_baseline` in prod

In [ ]:
%sql
-- capability_silence — WARN-tier liveness check: has each registered capability produced output
-- within its silence_grace_hours window? Joins capability_registry (active, non-null grace) against
-- v_llm_bronze (where output_type is aliased as capability, generated_at as called_at).
-- Prod capabilities: saa-display (2h), situational-awareness (2h), summary (36h).
-- sev2-insights has silence_grace_hours = NULL (irregular cadence) and is excluded by the WHERE.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_silence AS
SELECT
    r.capability, r.expected_min_daily, r.silence_grace_hours, r.owner,
    count(b.id)      AS calls_last_7d,
    max(b.called_at) AS last_call_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                     AS hours_since_last_call
FROM mq_gmdf_dev.oil_obs.capability_registry r
LEFT JOIN mq_gmdf_dev.oil_obs.v_llm_bronze b
       ON b.capability = r.capability
      AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
WHERE r.active = true AND r.silence_grace_hours IS NOT NULL
GROUP BY 1, 2, 3, 4
HAVING max(b.called_at) IS NULL
    OR unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at))
       > r.silence_grace_hours * 3600;

In [ ]:
%sql
-- shift_context_missing — WARN-tier data-quality check: detects output records where shift
-- context fields (shift_date, shift_type, batch_nbr) are blank or null. These fields enable
-- per-shift and per-batch slicing and AI-to-ISH correlation. Reads v_llm_bronze (where
-- output_type is aliased as capability, generated_at as called_at). Scoped to active
-- capabilities via capability_registry inner join.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.shift_context_missing AS
SELECT
    b.capability,
    count(*)                                                 AS total_calls,
    count_if(coalesce(b.shift_type, '') = '')                AS blank_shift_type,
    count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') AS blank_batch_nbr,
    count_if(b.shift_date IS NULL)                           AS null_shift_date,
    max(b.called_at)                                         AS last_seen,
    current_timestamp()                                      AS detected_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY b.capability
HAVING count_if(coalesce(b.shift_type, '') = '') > 0
    OR count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') > 0
    OR count_if(b.shift_date IS NULL) > 0;

In [ ]:
%sql
-- etl_pipeline_health — CRITICAL detector: captures any ETL task failure in the last 24 hours.
-- The upstream ETL refreshes ~19 source tables every 10-15 min. When a task fails, the SAA
-- agent continues producing outputs using stale data — capability_silence and pipeline_heartbeat
-- won't fire because the agent is still generating, making this the only early warning.
-- finding_signature keyed on (table_or_view, run_id) so each distinct failure is one incident.
-- Reads v_etl_bronze (pass-through view over mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit).
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_pipeline_health AS
SELECT
    run_id,
    run_timestamp,
    table_or_view,
    status,
    error_message,
    duration_seconds,
    sha2(concat_ws('|', table_or_view, run_id), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.v_etl_bronze
WHERE status = 'failure'
  AND run_timestamp >= current_timestamp() - INTERVAL 24 HOURS;